# Week 41, Day 2: Noise, accuracy and speed

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w041/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Constant coefficient of variation

For each command level, simulate 2,000 trials of force = u (1 + k e) for signal-dependent noise and force = u + s e for additive noise (e standard normal), and compute the coefficient of variation (standard deviation over mean) of each.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(1998)
levels = np.array([1.0, 2.0, 5.0, 10.0, 20.0])
k, s = 0.1, 0.5
cv_sdn, cv_add = [], []
for u in levels:
    e = r.normal(size=2000)
    f_sdn = ...
    f_add = ...
    cv_sdn.append(f_sdn.std() / f_sdn.mean()); cv_add.append(f_add.std() / f_add.mean())
cv_sdn, cv_add = np.array(cv_sdn), np.array(cv_add)
print("CV, signal-dependent:", np.round(cv_sdn, 3)); print("CV, additive:        ", np.round(cv_add, 3))
plt.plot(levels, cv_sdn, "o-", label="signal-dependent"); plt.plot(levels, cv_add, "s-", label="additive")
plt.xlabel("command"); plt.ylabel("coefficient of variation"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_sdn():
    "with signal-dependent noise, the coefficient of variation is the same (within 10%) at all command levels"
    assert cv_sdn.max() / cv_sdn.min() < 1.1

def test_additive():
    "with additive noise, the coefficient of variation falls more than fivefold from the weakest to the strongest command"
    assert cv_add[0] / cv_add[-1] > 5

test_sdn()
test_additive()
print("All checks passed")

### Exercise: Commands that minimise endpoint variance

Complete plan(D, T): build C (3 by T, the influence of each command on the final state) and a (the summed squared influence of each command on position over a 40-step hold), then solve for the commands with the formula above. Return the commands, the trajectory and the predicted standard deviation of position during the hold.

In [ ]:
dt, tau, k_noise, hold = 0.005, 0.04, 0.3, 40
A = np.array([[1, dt, 0], [0, 1, dt], [0, 0, 1 - dt / tau]]); B = np.array([0, 0, dt / tau])
P = [B]
for _ in range(600):
    P.append(A @ P[-1])
P = np.array(P)                                      # P[j] = A^j B: effect of a command j steps later

def plan(D, T):
    t = np.arange(T)
    C = P[T - 1 - t].T                                # 3 x T: effect of each command on the state at the end
    W = np.array([P[T - 1 - t + h, 0] for h in range(1, hold + 1)])   # hold x T: effect on later positions
    a = k_noise ** 2 * (W ** 2).sum(0) + 1e-15
    b = np.array([D, 0.0, 0.0])
    u = ...
    x, traj = np.zeros(3), []
    for ut in u:
        x = A @ x + B * ut; traj.append(x.copy())
    sd = np.sqrt(np.sum(a * u ** 2) / hold)
    return u, np.array(traj), sd

u, traj, sd = plan(0.1, 80)
print(f"end position {traj[-1, 0]:.4f} m, end velocity {traj[-1, 1]:.2e}; predicted endpoint spread {sd * 1000:.2f} mm")
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
tt = np.arange(80) * dt
ax[0].plot(tt, traj[:, 1]); ax[0].set_title("velocity (m/s)"); ax[1].plot(tt, u); ax[1].set_title("command")
for a_ in ax: a_.set_xlabel("s")
plt.show()

In [ ]:
import numpy as np
def test_target():
    "the planned movement reaches 10 cm and stops there"
    assert abs(traj[-1, 0] - 0.1) < 1e-6 and abs(traj[-1, 1]) < 1e-6

def test_shape():
    "the velocity profile is single-peaked and positive throughout"
    v = traj[:, 1]
    assert np.all(v > -1e-9) and np.sum((v[1:-1] > v[:-2]) & (v[1:-1] > v[2:])) == 1

def test_scaling():
    "endpoint spread doubles when the distance doubles"
    assert abs(plan(0.2, 80)[2] / plan(0.1, 80)[2] - 2) < 1e-6

test_target()
test_shape()
test_scaling()
print("All checks passed")

### Exercise: A Fitts-like law from noise

Because the spread scales exactly with distance, compute sd_unit[T], the spread for a 1 m reach, once for each duration. Then for each D and W, the movement time is the shortest duration with 4 D sd_unit[T] at most W. Fit MT = a + b ID by least squares and report the slope and R².

In [ ]:
Ts = np.arange(20, 400, 5)
sd_unit = np.array([plan(1.0, T)[2] for T in Ts])
ID, MT = [], []
for D in (0.05, 0.1, 0.2, 0.4):
    for W in (0.01, 0.02, 0.04):
        ok = ...
        if len(ok):
            ID.append(np.log2(2 * D / W)); MT.append(ok[0] * dt)
ID, MT = np.array(ID), np.array(MT)
slope, intercept = np.polyfit(ID, MT, 1)
r2 = 1 - np.sum((MT - (intercept + slope * ID)) ** 2) / np.sum((MT - MT.mean()) ** 2)
print(f"MT = {intercept:.3f} + {slope:.3f} x ID seconds, R squared {r2:.2f}")
plt.plot(ID, MT, "o"); plt.plot(np.sort(ID), intercept + slope * np.sort(ID)); plt.xlabel("index of difficulty (bits)"); plt.ylabel("movement time (s)"); plt.show()

In [ ]:
import numpy as np
def test_fit():
    "movement time increases with index of difficulty, and a straight line explains more than 80% of the variance"
    assert slope > 0 and r2 > 0.8

test_fit()
print("All checks passed")